# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Asizem Curtis
**Student ID:** 54792028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [36]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [6]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
response = ''
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
    global response
    response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},],
                temperature=temperature,
                max_tokens=max_tokens,)
    return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
simple_question_response = ask_llm("What is the most figure for the human population")
print(simple_question_response)
# TODO: Print response.usage as well — how many tokens did your call consume?
print()
print(f'Prompt token usage: {response.usage.prompt_tokens}')
print(f'Completion token usage: {response.usage.completion_tokens}')
print(f'Token token usage:{response.usage.total_tokens}')

The most recent figure for the human population is approximately 7.92 billion people, according to the United Nations Department of Economic and Social Affairs (UN DESA) estimates for mid-2023.

Here's a rough breakdown of the world population growth:

* 1800: 1 billion
* 1927: 2 billion
* 1960: 3 billion
* 1974: 4 billion
* 1987: 5 billion
* 1999: 6 billion
* 2011: 7 billion
* 2023: 7.92 billion (estimated)

The world population is projected to continue growing, albeit at a slower rate, and is expected to reach:

* 8 billion by 2024
* 9 billion by 2037
* 10 billion by 2058
* 11 billion by 2100 (according to the medium-variant projection)

Please note that these numbers are estimates and may vary slightly depending on the source and methodology used.

Prompt token usage: 50
Completion token usage: 212
Token token usage:262


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. The system role is defined by the developer, provides the model with guidelines for behaviour and constraints for the overall conversation but the user role is defined by the user, serves as a basis/input for the model's reponse and may be limited a specific question-answer turn in the conversation. An example of something in the system is the role,  but for the user role is the question.

2. A token is a portion of the text that is processed by the model. This is typically a single word for English, but longer words may be split into several tokens.
Because computational resources required grow for the number of tokens in the request or query, the cost is calculated based on tokens. If cost is calculated based on requests, some requests may require more computational resources by virtue of the fact they have more tokens.

### Part 1.2 — Temperature: the randomness dial

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.
temperatures = [0.0, 1.2]
for i in temperatures:
  print(f"---Temperature at {i}---")
  for j in range(5):
    print(f"request {j}: {ask_llm("Suggest a single name for a savings product for market traders in Accra.", temperature=i)}")

---Temperature at 0.0---
request 0: "Makola Save" would be a fitting name for a savings product for market traders in Accra. "Makola" is a well-known market in Accra, and the name immediately conveys that the product is tailored to the needs of market traders, while "Save" clearly communicates the savings aspect of the product.
request 1: "Makola Save" would be a fitting name for a savings product for market traders in Accra. "Makola" is a well-known market in Accra, and the name immediately conveys that the product is tailored to the needs of market traders, while "Save" clearly communicates the savings aspect of the product.
request 2: "Makola Save" would be a fitting name for a savings product for market traders in Accra. "Makola" is a well-known market in Accra, and the name immediately conveys that the product is tailored to the needs of market traders, while "Save" clearly communicates the savings aspect of the product.
request 3: "Makola Save" would be a suitable name for a savi

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**
At temperature 0.0, the responses produced are almost entirely the same. In the other words, the responses are mostly consistent, very predictable and the language is very regular. At temperature 1.2, the responses produced are entirely dissimilar. Each response cannot be predicted based on the based on the previous ones. The response display increasing creativity compared to the responses at 0.0 temperature. For a loan decision-support system, the most important for an support system is consistency, to be able to justify their decision at any point. Using temperature range close to 0.0 to 0.3 allows this consistency, because it produces almost the same response

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
responses_v1 = {}
SUMMARY_PROMPT_V1 = "Summarize this: \n\n"
for i in ['L002', 'L006']:
  responses_v1[i] = ask_llm(SUMMARY_PROMPT_V1+LETTERS[i])


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
responses_v2 = {}
SUMMARY_PROMPT_V2 = {'system_prompt': "You are an assistant to a microfinance loan officer. Your objective is to summarize a loan application within 1-2 sentences with embellishments and neutrally",
                     'user_prompt': "Summarize this loan application: \n\n"}
for i in ['L002', 'L006']:
  responses_v2[i] = ask_llm(SUMMARY_PROMPT_V2['user_prompt']+LETTERS[i], system_prompt=SUMMARY_PROMPT_V2['system_prompt'], temperature=0)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("----For summary L002: -----")
print("V1:\n" + responses_v1['L002'])
print()
print("V2:\n" + responses_v2['L002'])

print("----For summary L002: -----")
print("V1:\n" + responses_v1['L006'])
print()
print("V2:\n" + responses_v2['L006'])


----For summary L002: -----
V1:
Kwame Boateng, a commercial driver in Kumasi, is seeking an urgent loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's struggling due to slow business, but expects an improvement after the festive season and promises to repay the loan as soon as possible, despite not having collateral.

V2:
Kwame Boateng, a commercial driver from Kumasi, has applied for a GHS 25,000 loan to repair his vehicle's engine and settle personal debts, anticipating an upturn in business after the festive season. He is seeking urgent assistance, although currently without collateral, and is relying on future earnings to repay the loan.
----For summary L002: -----
V1:
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his b

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V2 is shown to use a consistent language style and number in both letters. This is not observed in V1. For example, V2 maintains sentence length of 2, with no use of punctuation other full stops and commas in both responses. The sentence and speech style is more formal than for V1, which is what one might expect from professional work setting. This is seen in the use of language such as "projected business growth", "future earnings", and "business acumen"

2. This is meant to prevent the model from including factually incorrect or plausible-sounding but non-grounded information in its responses. This is known in LLM literature as "hallucination".

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [22]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_PROMPT_SYSTEM = """You are data extraction implement. Your task is to process letter requests for
                    loan applications and convert them into a valid JSON obejct with the below specified
                    keys. You will extract information present in the letter request, with no explanation,
                    markdown formatting or code fence
                    -applicant_name(string)
                    -amount_ghs(number)
                    -purpose(string)
                    -monthly_profit_ghs(number or null if not present)
                    -has_collateral_or_guaranteed(boolean)
                    -repayment_months(number or null if not present)

                    For example,
                    Letter:
                    "Dear Manager,
                     My name is Ama Kwaku. I sell cooked food near the Kwabenya station. I am requesting a loan of
                     GHS 1,800 to buy a new gas cylinder and cooking equipment. My monthly profit is about GHS 600.
                     My sister shall serve as my guarantor. I plan to repay the loan within a 5-month period.

                    Output:
                    {{"applicant_name": "Ama Serwaa", "amount_ghs" : 1800, "purpose": "buy a new gas cylinder and cooking equipment", "monthly_profit": 600, "has_collateral_or_guarantor": true, "repayment_months": 5}}

                    If it does not exist, do not use anything other than null.
                 """
EXTRACT_PROMPT_USER = "Extract this letter: \n"


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
import json

def extract_fields(letter_text):
  prompt = EXTRACT_PROMPT_USER+ letter_text
  raw_output = ask_llm(user_prompt=prompt, system_prompt=EXTRACT_PROMPT_SYSTEM, temperature=0)
  clean_output = raw_output.strip()

  if raw_output.startswith("```"):
    clean_output = raw_output.strip("```")

  try:
    data = json.loads(clean_output)
    return data
  except json.JSONDecodeError as e:
    print(f"Failed to parse as JSON: {e}")
    print(f"Raw output was \n{repr(raw_output)}\n")


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
import pandas as pd

extracted_data = {}
extracted_data = pd.DataFrame(extracted_data)

row = []
for letter in LETTERS.keys():
  result = extract_fields(LETTERS[letter])

  if result is not None:
    extracted_data[letter] = result
  else:
    print("Parsing failed")

extracted_data_transposed = extracted_data.transpose()
extracted_data_transposed.head(6)

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guaranteed,repayment_months
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900,True,20
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,None,False,None
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800,True,15
L004,Yaw Owusu,12000,for feed and 500 new layers,1500,True,18
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,None,True,16
L006,Kofi,50000,"start a car washing business, a provision shop...",None,False,12


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. Using the letters present in the test undermines the test. When this done, there is no guarantee that the model has understood the pattern, and is simply producing a memorized outcome. This raises the accuracy and makes the system look like it performs well in practicality.
2. This is to prevent hallucination. If this were not specified, it would likely find plausible data to replace fields it could not find in the letter.
3. temperature=0 makes it so it the results are always the same, in other words they are repeatable and consistent. For extractive tasks, it ensures the data is always represented. However for more creative tasks, this repeatitive is not ideal.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [39]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT_SYSTEM = """You are assistant at a microfinance institution. Your task is
                         prepare a objective recommendation brief when provided a letter request for
                         a loan and JSON extracted object concerning the loan.

                         OUTPUT REQUIREMENTS:
                         -You must provide the strengths of the application.
                         -You must provide the risks of the application.
                         -You must provide the officer in charge with information missing from the
                          letter that must be requested
                         -You must provide an suggestion for the next step such as "request documents",
                          "invite for interview", "flag for senior review"

                         You must ground the brief in the letters. There can be no extrapolation, or guesses.
                         You must the points of each output requirement in bullet points. You cannot provide final
                         decision suggestions such as "approve" or "reject". It should be written in proper language
                         without "\n", "\'", etc programming constructs.
               """
BRIEF_USER_PROMPT = "Provide the brief for the loan officer using this: \n"
# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
extracted_briefs = {}
for letter in LETTERS.keys():
  prompt = BRIEF_USER_PROMPT + f"Letter:\n {LETTERS[letter]} \n Extracted JSON: \n{extracted_data[letter]}"

  extracted_briefs[letter] = ask_llm(user_prompt=prompt, system_prompt=BRIEF_PROMPT_SYSTEM)
for i in ["L001", "L002", "L006"]:
   print(f"{i} Brief: \n{extracted_briefs[i]}\n")

print(f"L003 Brief: \n{extracted_briefs["L003"]}\n")


L001 Brief: 
Objective Recommendation Brief

**Strengths of the Application:**
* The applicant, Akosua Mensah, has a established business with 12 years of experience selling provisions at Makola Market.
* The applicant has a consistent profit record, with a monthly profit of GHS 900.
* The applicant has demonstrated a good savings history, having saved GHS 2,500 over two years with the susu scheme without missing a contribution.
* The applicant has a guarantor, her sister, who is a teacher, which provides an additional layer of security for the loan.
* The applicant has a clear plan for the loan, with a specific purpose (buying a deep freezer and expanding into frozen foods) and a repayment plan (GHS 450 monthly over 20 months).

**Risks of the Application:**
* The applicant is requesting a significant loan amount (GHS 8,000) which may pose a risk if the business is not able to generate sufficient profits to meet the repayment obligations.
* The repayment period (20 months) is relative

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. For L003, the system expresses the stability of the business and revenue, the presence of collateral, the plausibilty of the repayment plan as strengths and the significant risky loan amount, the expection of a single season covering a significant portion of the revenue, and the lack of mention of experience as weakness which is right in my opinion. For L006, the system expresses the confidence, enthusiasm, youthfulness, as well as the diversity and lucrativeness of the stated businesses as strengths, and for weaknesses, it expresses the lack of experience, absence of collateral and general lack of information, as weaknesses, which I believe is also a correct evaluation.

2. The model has little solid understanding of the real-world or context surrounding an application.It can also perpetuate bias in its decision-making. It has no way of assessing the trustworthiness of a claim in an application and so it is not suitable as final decision-maker.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.